In [172]:
#TRACKED PARCEL STATISTICS
#MAINLY HISTOGRAMS OF CI LOCATION (T,Z)

In [403]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
res='1km'
Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***


job_array=False;index_adjust=0
ocean_fraction=0.25

In [404]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [365]:
#SETTING UP JOB_ARRAY
###################################
job_array=False
job_array=True

In [366]:
#JOB ARRAY SETUP
if job_array==True:

    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=2
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

In [367]:
if job_array==True:
    #Indexing Array with JobArray
    parcel=parcel.isel(xh=slice(start_job,end_job))
    #(for 150_000_000 parcels use 500-1000 jobs)

In [320]:
################################################################################

In [405]:
def find_SBZ_xmaxs():
    # Define the directory and file path
    dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
    file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}_{Np_str}_5min' + '.h5'
    
    # Open the HDF5 file in read mode
    with h5py.File(file_path, 'r') as f:
        # Access the 'conv' dataset
        conv_dataset = f['conv']
        
        # Define the vertical level you are interested in
        zlev = 4
        
        # Initialize a list to store the xmaxs for each time step
        xmaxs_list = []

        # Loop over each time step (axis=0 corresponds to time)
        for t in range(conv_dataset.shape[0]):  # conv_dataset.shape[0] is the time dimension size
            # Read the relevant slice for this time step and vertical level
            Conv_t_zlev = conv_dataset[t, zlev, :, :]  # Shape should be (y_size, x_size)
            
            # Calculate the mean across the y-axis
            Conv_ymean = np.mean(Conv_t_zlev, axis=0)  # Mean across the y-axis
            
            # Find the index of the maximum value along the x-axis
            xmax = np.argmax(Conv_ymean)
            
            # Append the result for this time step
            xmaxs_list.append(xmax)
    
    # Convert the list of xmaxs to a numpy array (optional)
    xmaxs = np.array(xmaxs_list)

    return xmaxs #returns SBZ x location for each timestep

SBZ_xmaxs=find_SBZ_xmaxs()

In [330]:
#RUNNING
##################################

In [331]:
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'

########################################################################

Ps=ALL_out_arr[:,0]
Ts=ALL_out_arr[:,1]

Coast_x=int(len(data['xh'])*2/8)
Coast_Lst=[]
SBZ_Lst=[]

kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1) #number of kms per grid
Position=(Ts,Ps)

with h5py.File(in_file, 'r') as f:
    for count,(t,p) in enumerate(zip(*Position)):
        if np.mod(count,3000)==0: print(f"{count}/{len(Position[0])}")
        Coast_Dist=f['X'][t,p]-Coast_x
        Coast_Dist*=kms
        Coast_Lst.append(Coast_Dist)
        
        SBZ_Dist=f['X'][t,p]-SBZ_xmaxs[t]
        SBZ_Dist*=kms
        SBZ_Lst.append(SBZ_Dist)

# plt.hist(SBZ_Lst,bins=300)
# plt.ylabel('count');plt.xlabel('X Distance (km)')
# plt.title('Distance from SBZ Front')
# plt.xlim((-256,256))

# plt.hist(Coast_Lst,bins=300)
# plt.ylabel('count');plt.xlabel('X Distance (km)')
# plt.title('Distance from Coast')
# plt.xlim((-256,256))

0/251


In [ ]:
########################
#READING BACK IN
def LoadFinalData(in_file):
    dict = {}
    with h5py.File(in_file, 'r') as f:
        for key in f.keys():
            dict[key] = f[key][:]
    return dict

def LoadAllCloudBase():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"all_cloudbase_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        all_cloudbase = pickle.load(f)
    return(all_cloudbase)
min_all_cloudbase=np.nanmin(LoadAllCloudBase())
print(f"Minimum Cloudbase is: {min_all_cloudbase}\n")

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
in_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
final_dict=LoadFinalData(in_file)


#DYNAMICALLY CREATING VARIABLES
for key, value in final_dict.items():
    globals()[key] = value

# #DYNAMICALLY PRINTING VARIABLE SIZES
# for key in final_dict:
#     print(f"{key} has {final_dict[key].shape[0]} parcels")

# PRINTING VARIABLE SIZES (ONE BY ONE)
print(f'ALL: {len(CL_ALL_out_arr)} CL parcels and {len(nonCL_ALL_out_arr)} nonCL parcels')
print(f'SHALLOW: {len(CL_SHALLOW_out_arr)} CL parcels and {len(nonCL_SHALLOW_out_arr)} nonCL parcels')
print(f'DEEP: {len(CL_DEEP_out_arr)} CL parcels and {len(nonCL_DEEP_out_arr)} nonCL parcels')
print('\n')
print(f'ALL: {len(SBZ_ALL_out_arr)} SBZ parcels and {len(nonSBZ_ALL_out_arr)} nonSBZ parcels')
print(f'SHALLOW: {len(SBZ_SHALLOW_out_arr)} SBZ parcels and {len(nonSBZ_SHALLOW_out_arr)} nonSBZ parcels')
print(f'DEEP: {len(SBZ_DEEP_out_arr)} SBZ parcels and {len(nonSBZ_DEEP_out_arr)} nonSBZ parcels')
print('\n')
print(f'ALL: {len(ColdPool_ALL_out_arr)} ColdPool parcels')
print(f'SHALLOW: {len(ColdPool_SHALLOW_out_arr)} ColdPool parcels')
print(f'DEEP: {len(ColdPool_DEEP_out_arr)} ColdPool parcels')


#APPLYING JOB ARRAY
if "job_array" in globals():
    print('APPLYING JOB ARRAY')
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    for name in [
        'CL_ALL_out_arr', 'CL_ALL_out_arr',
        'CL_SHALLOW_out_arr', 'CL_SHALLOW_out_arr',
        'CL_DEEP_out_arr', 'nonCL_DEEP_out_arr',
        'SBZ_ALL_out_arr', 'nonSBZ_ALL_out_arr',
        'SBZ_SHALLOW_out_arr', 'nonSBZ_SHALLOW_out_arr',
        'SBZ_DEEP_out_arr', 'nonSBZ_DEEP_out_arr',
        'ColdPool_ALL_out_arr', 'ColdPool_SHALLOW_out_arr', 'ColdPool_DEEP_out_arr'
    ]:
        globals()[name] = job_filter(globals()[name])

In [332]:
#SAVING
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out/'
out_file=dir2+f"ALL_Tracked_Parcel_Stats_Lists_5min_{job_id}.pkl"
with open(out_file, 'wb') as f:
    pickle.dump({'SBZ_Lst': SBZ_Lst, 'Coast_Lst': Coast_Lst}, f)

In [377]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [379]:

if recombine==True:
    # SBZ_Lst=[]
    # Coast_Lst=[]
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out/'
    
    num_jobs=60
    for job_id in range(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(job_id)
        #READING BACK IN
        in_file=dir2+f"ALL_Tracked_Parcel_Stats_Lists_5min_{job_id}.pkl"
        with open(in_file, 'rb') as f:
            load_data = pickle.load(f)
            SBZ_Lst += load_data['SBZ_Lst']
            Coast_Lst += load_data['Coast_Lst']

    #SAVING
    dir3=dir+'Project_Algorithms/Tracked_Profiles/'
    out_file=dir3+f"ALL_Tracked_Parcel_Stats_Lists_5min.pkl"
    with open(out_file, 'wb') as f:
        pickle.dump({'SBZ_Lst': SBZ_Lst, 'Coast_Lst': Coast_Lst}, f)


10
20
30
40
50
60


In [ ]:
#RUNNING
##############################

In [ ]:
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'

####################################################################################

Ps=SHALLOW_out_arr[:,0]
Ts=SHALLOW_out_arr[:,1]

Coast_x=int(len(data['xh'])*2/8)
Coast_Lst=[]
SBZ_Lst=[]

kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1) #number of kms per grid
Position=(Ts,Ps)
with h5py.File(in_file, 'r') as f:
    for count,(t,p) in enumerate(zip(*Position)):
        if np.mod(count,3000)==0: print(f"{count}/{len(Position[0])}")
        Coast_Dist=f['X'][t,p]-Coast_x
        Coast_Dist*=kms
        Coast_Lst.append(Coast_Dist)
        
        SBZ_Dist=f['X'][t,p]-SBZ_xmaxs[t]
        SBZ_Dist*=kms
        SBZ_Lst.append(SBZ_Dist)

# plt.hist(SBZ_Lst,bins=300)
# plt.ylabel('count');plt.xlabel('X Distance (km)')
# plt.title('Distance from SBZ Front')
# plt.xlim((-256,256))

# plt.hist(Coast_Lst,bins=300)
# plt.ylabel('count');plt.xlabel('X Distance (km)')
# plt.title('Distance from Coast')
# plt.xlim((-256,256))

In [339]:
#SAVING
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out/'
out_file=dir2+f"SHALLOW_Tracked_Parcel_Stats_Lists_5min_{job_id}.pkl"
with open(out_file, 'wb') as f:
    pickle.dump({'SBZ_Lst': SBZ_Lst, 'Coast_Lst': Coast_Lst}, f)

In [340]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [384]:
if recombine==True:
    # SBZ_Lst=[]
    # Coast_Lst=[]
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out/'
    
    num_jobs=60
    for job_id in range(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(job_id)
        #READING BACK IN
        in_file=dir2+f"SHALLOW_Tracked_Parcel_Stats_Lists_5min_{job_id}.pkl"
        with open(in_file, 'rb') as f:
            load_data = pickle.load(f)
            SBZ_Lst += load_data['SBZ_Lst']
            Coast_Lst += load_data['Coast_Lst']

    #SAVING
    dir3=dir+'Project_Algorithms/Tracked_Profiles/'
    out_file=dir3+f"SHALLOW_Tracked_Parcel_Stats_Lists_5min.pkl"
    with open(out_file, 'wb') as f:
        pickle.dump({'SBZ_Lst': SBZ_Lst, 'Coast_Lst': Coast_Lst}, f)


10
20
30
40
50
60


In [ ]:
#RUNNING
##############################

In [ ]:
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'

####################################################################################

Ps=DEEP_out_arr[:,0]
Ts=DEEP_out_arr[:,1]

Coast_x=int(len(data['xh'])*2/8)
Coast_Lst=[]
SBZ_Lst=[]

kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1) #number of kms per grid
Position=(Ts,Ps)
with h5py.File(in_file, 'r') as f:
    for count,(t,p) in enumerate(zip(*Position)):
        if np.mod(count,3000)==0: print(f"{count}/{len(Position[0])}")
        Coast_Dist=f['X'][t,p]-Coast_x
        Coast_Dist*=kms
        Coast_Lst.append(Coast_Dist)
        
        SBZ_Dist=f['X'][t,p]-SBZ_xmaxs[t]
        SBZ_Dist*=kms
        SBZ_Lst.append(SBZ_Dist)

# plt.hist(SBZ_Lst,bins=300)
# plt.ylabel('count');plt.xlabel('X Distance (km)')
# plt.title('Distance from SBZ Front')
# plt.xlim((-256,256))

# plt.hist(Coast_Lst,bins=300)
# plt.ylabel('count');plt.xlabel('X Distance (km)')
# plt.title('Distance from Coast')
# plt.xlim((-256,256))

In [347]:
#SAVING
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out/'
out_file=dir2+f"DEEP_Tracked_Parcel_Stats_Lists_5min_{job_id}.pkl"
with open(out_file, 'wb') as f:
    pickle.dump({'SBZ_Lst': SBZ_Lst, 'Coast_Lst': Coast_Lst}, f)

In [348]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [421]:
if recombine==True:
    # SBZ_Lst=[]
    # Coast_Lst=[]
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out/'
    
    num_jobs=60
    for job_id in range(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(job_id)
        #READING BACK IN
        in_file=dir2+f"DEEP_Tracked_Parcel_Stats_Lists_5min_{job_id}.pkl"
        with open(in_file, 'rb') as f:
            load_data = pickle.load(f)
            SBZ_Lst += load_data['SBZ_Lst']
            Coast_Lst += load_data['Coast_Lst']

    #SAVING
    dir3=dir+'Project_Algorithms/Tracked_Profiles/'
    out_file=dir3+f"DEEP_Tracked_Parcel_Stats_Lists_5min.pkl"
    with open(out_file, 'wb') as f:
        pickle.dump({'SBZ_Lst': SBZ_Lst, 'Coast_Lst': Coast_Lst}, f)


10
20
30
40
50
60


In [353]:
#RUNNING
####################################################
#WHEN TRACKED PARCELS EXIT CLOUDY REGION

In [355]:
#GET AFTER ARRAYS
ALL_out_after_array=ALL_out_arr[:,3]
SHALLOW_out_after_array=SHALLOW_out_arr[:,3]
DEEP_out_after_array=DEEP_out_arr[:,3]


#AFTER TIME NEEDS TO BE ADDED TO FINAL TIMESTEPS IN OUT_ARR
ALL_out_after_array+=(ALL_out_arr[:,2]+1-ALL_out_arr[:,1]).astype(int)
SHALLOW_out_after_array+=(SHALLOW_out_arr[:,2]+1-SHALLOW_out_arr[:,1]).astype(int)
DEEP_out_after_array+=(DEEP_out_arr[:,2]+1-DEEP_out_arr[:,1]).astype(int)

In [356]:
#SAVING
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out/'
out_file=dir2+f"After_Arrays_5min_{job_id}.npz"
np.savez(out_file,
         ALL=ALL_out_after_array,
         SHALLOW=SHALLOW_out_after_array,
         DEEP=DEEP_out_after_array)

In [357]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [394]:
if recombine == True:
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out/'
    
    # Initialize empty lists to collect arrays
    ALL_list = []
    SHALLOW_list = []
    DEEP_list = []

    num_jobs = 60
    for job_id in range(1, num_jobs + 1):
        if np.mod(job_id, 10) == 0: print(f"Loading job {job_id}")

        in_file = dir2 + f"After_Arrays_5min_{job_id}.npz"
        with np.load(in_file) as data:
            ALL_list.append(data['ALL'])
            SHALLOW_list.append(data['SHALLOW'])
            DEEP_list.append(data['DEEP'])

    # Concatenate all arrays
    ALL_combined = np.concatenate(ALL_list, axis=0)
    SHALLOW_combined = np.concatenate(SHALLOW_list, axis=0)
    DEEP_combined = np.concatenate(DEEP_list, axis=0)

    # Save to single .npz file
    dir3 = dir + 'Project_Algorithms/Tracked_Profiles/'
    out_file = dir3 + "After_Arrays_5min.npz"
    np.savez(out_file,
             ALL=ALL_combined,
             SHALLOW=SHALLOW_combined,
             DEEP=DEEP_combined)

    # print(f"Saved combined arrays to: {out_file}")


Loading job 10
Loading job 20
Loading job 30
Loading job 40
Loading job 50
Loading job 60
